# Analiza cech i klasyfikacja autentyczności obrazów

Notebook laboratoryjny: ELA, DCT, statystyki szumu, embeddingi patchy ResNet-18 oraz Gradient Boosting z LOOCV. Nie jest gotowym detektorem ani dowodem autentyczności zdjęcia.

## Uruchomienie

1. Użyj osobnego środowiska Python z NumPy, pandas, Pillow, Matplotlib, OpenCV, PyTorch, torchvision i scikit-learn oraz JupyterLab.
2. Otwórz notebook z katalogu Background modeling. Jedyna konfiguracja to DATASET_DIR w pierwszej komórce, domyślnie Dataset.
3. Przygotuj własne obrazy w podkatalogach real/ i tampered/: minimum dwa poprawne obrazy każdej klasy, co najmniej 32 x 32 piksele. Obsługiwane są JPG, JPEG, PNG, BMP, TIF i TIFF.
4. Wykonaj Restart Kernel and Run All. Pierwsza inicjalizacja ResNet-18 może pobrać wagi przez torchvision do lokalnego cache. CUDA jest opcjonalna; CPU również jest obsługiwane.

Zdjęcia i wagi nie są częścią repozytorium. Używaj wyłącznie danych, które możesz przetwarzać zgodnie z ich warunkami. Do uruchomienia nie jest potrzebna kamera. Wersja publikowana nie zawiera zapisanych obrazów, tabel wynikowych ani liczników wykonania. Historyczne obserwacje na końcu nie są wynikiem uruchomienia przez nowego użytkownika.

In [ ]:
from io import BytesIO
from pathlib import Path

import numpy as np
from PIL import Image, ImageChops, ImageEnhance
import matplotlib.pyplot as plt
import cv2 as cv

# Jedyna konfiguracja użytkownika: katalog zawierający real/ oraz tampered/.
# Uruchamiaj notebook z katalogu Background modeling lub podaj własną ścieżkę.
DATASET_DIR = Path("Dataset")
real_folder = DATASET_DIR / "real"
tampered_folder = DATASET_DIR / "tampered"
MIN_IMAGE_SIZE = 32
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def read_dataset_image(image_path, flags=cv.IMREAD_COLOR):
    path = Path(image_path)
    try:
        image = cv.imdecode(np.frombuffer(path.read_bytes(), dtype=np.uint8), flags)
    except (OSError, cv.error) as error:
        raise ValueError(f"Nie można odczytać obrazu: {path}") from error
    if image is None:
        raise ValueError(f"Nie można odczytać obrazu: {path}")
    height, width = image.shape[:2]
    if min(height, width) < MIN_IMAGE_SIZE:
        raise ValueError(
            f"Obraz {path} ma {width} x {height} pikseli; "
            f"wymagane minimum to {MIN_IMAGE_SIZE} x {MIN_IMAGE_SIZE}."
        )
    return image


def list_dataset_images(folder):
    directory = Path(folder)
    if not directory.is_dir():
        raise FileNotFoundError(
            f"Brak katalogu {directory}. Ustaw DATASET_DIR na zbiór z real/ i tampered/."
        )
    paths = sorted(
        (path for path in directory.iterdir()
         if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES),
        key=lambda path: path.name,
    )
    if len(paths) < 2:
        raise ValueError(
            f"Katalog {directory} musi zawierać minimum dwa obrazy każdej klasy dla LOOCV."
        )
    for path in paths:
        read_dataset_image(path)
    return paths


def show_image_grid(images, titles, columns=5):
    if len(images) != len(titles):
        raise ValueError("Liczba podpisów musi odpowiadać liczbie obrazów.")
    if columns < 1:
        raise ValueError("Liczba kolumn musi być dodatnia.")
    if len(images) == 0:
        print("Brak obrazów do wyświetlenia.")
        return None
    rows = (len(images) + columns - 1) // columns
    figure, axes = plt.subplots(
        rows, columns, figsize=(15, 3 * rows), squeeze=False
    )
    for axis in axes.flat:
        axis.axis("off")
    for index, (image, title) in enumerate(zip(images, titles, strict=True)):
        axes.flat[index].imshow(image)
        axes.flat[index].set_title(title, fontsize=8)
    figure.tight_layout()
    plt.show()
    return figure

In [ ]:
tampered_paths = list_dataset_images(tampered_folder)
tampered_images_dirs = [path.name for path in tampered_paths]
print(f"Liczba obrazów w folderze tampered: {len(tampered_paths)}")
print(tampered_images_dirs)

Wczytanie obrazów zmanipulowanych z katalogu, konwersja ich przestrzeni barw z BGR do RGB
oraz wyświetlenie ich w postaci siatki subplotów wraz z nazwami plików

In [ ]:
tampered_images = [
    cv.cvtColor(read_dataset_image(path), cv.COLOR_BGR2RGB)
    for path in tampered_paths
]
show_image_grid(tampered_images, tampered_images_dirs)

In [ ]:
real_paths = list_dataset_images(real_folder)
real_images_dirs = [path.name for path in real_paths]
print(f"Liczba obrazów w folderze real: {len(real_paths)}")
print(real_images_dirs)

Wczytanie obrazów niezmanipulowanych z katalogu, konwersja ich przestrzeni barw z BGR do RGB
oraz wyświetlenie ich w postaci siatki subplotów wraz z nazwami plików

In [ ]:
real_images = [
    cv.cvtColor(read_dataset_image(path), cv.COLOR_BGR2RGB)
    for path in real_paths
]
show_image_grid(real_images, real_images_dirs)

Sprawdzenie dostępności akceleracji GPU (CUDA) i ustawienie urządzenia obliczeniowego
na GPU, jeśli jest dostępne, w przeciwnym razie na CPU

In [ ]:
import torch
import torchvision.transforms as T
import torchvision.models as models
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
device

Inicjalizacja wstępnie wytrenowanego modelu ResNet-18, usunięcie warstwy klasyfikacyjnej <br>
w celu ekstrakcji wektorów cech (embeddingów), zdefiniowanie transformacji obrazu <br>
zgodnych z ImageNet oraz funkcji konwertującej obraz wejściowy do 512-wymiarowego <br>
wektora cech przy użyciu sieci neuronowej <br>

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device)
resnet.eval()

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

def _resnet_embed(img_in) -> np.ndarray:
    if isinstance(img_in, np.ndarray):
        arr = img_in
        if arr.ndim == 2:  # grayscale
            arr = np.stack([arr, arr, arr], axis=-1)
        elif arr.ndim == 3 and arr.shape[0] == 3:
            arr = np.transpose(arr, (1, 2, 0))
        img = Image.fromarray(arr.astype(np.uint8))
    elif isinstance(img_in, Image.Image):
        img = img_in.convert("RGB")
    else:
        raise TypeError(f"Unexpected type {type(img_in)}")

    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = resnet(x).squeeze(0)

    return feat.detach().cpu().numpy().astype(np.float32)

# Opis funkcji resnet_patch_features()
    Ekstrakcja cech obrazu w oparciu o patchowanie: obraz jest dzielony na siatkę
    (grid) patchy, z których każdy jest przetwarzany przez ResNet-18
    w celu uzyskania 512-wymiarowego embeddingu. Następnie dla wszystkich patchy obliczane
    są statystyki agregujące (średnia i maksimum) i łączone w jeden 1024-wymiarowy wektor
    cech reprezentujący cały obraz.

In [ ]:
def resnet_patch_features(image_path, grid=(3, 3)) -> np.ndarray:
    with Image.open(image_path) as original:
        img = original.convert("RGB")
    W, H = img.size
    gy, gx = grid

    patch_w = W // gx
    patch_h = H // gy

    feats = []
    for iy in range(gy):
        for ix in range(gx):
            left = ix * patch_w
            top = iy * patch_h
            right = left + patch_w
            bottom = top + patch_h

            patch = img.crop((left, top, right, bottom))
            feats.append(_resnet_embed(patch))

    feats = np.stack(feats, axis=0)  # (n_patches, 512)

    feat_mean = feats.mean(axis=0)   # (512,)
    feat_max = feats.max(axis=0)     # (512,)

    return np.concatenate([feat_mean, feat_max], axis=0).astype(np.float32)

Wczytanie obrazu i uzyskanie globalnego embeddingu (512-wymiarowego wektora cech)
z ResNet-18 bez dzielenia na patch-e.

In [ ]:

def resnet_global_features(image_path) -> np.ndarray:
    with Image.open(image_path) as original:
        img = original.convert("RGB")
    return _resnet_embed(img)

Ekstrakcja cech z obrazu za pomocą ResNet-18 na patchach: funkcja zwraca
wektor cech (1024), nazwy poszczególnych cech oraz metadane (grid i wymiar).

In [ ]:
def extract_features_patch_cnn(image_path, grid=(3, 3)):
    feature_values = []
    feature_names = []
    all_features = {}

    cnn_feat = resnet_patch_features(image_path, grid=grid)  # (1024,)
    feature_values.append(cnn_feat)
    feature_names += [f"cnn_patch_{i}" for i in range(cnn_feat.shape[0])]

    all_features["cnn"] = {"grid_y": grid[0], "grid_x": grid[1], "dim": int(cnn_feat.shape[0])}
    return feature_values, feature_names, all_features

# Analiza
# Opis funkcji perform_ela_gray()
    Wykonuje klasyczną analizę ELA (Error Level Analysis) dla obrazu JPEG.

    Procedura:
    1. Wczytuje obraz i konwertuje do RGB.
    2. Zapisuje obraz ponownie jako JPEG o zadanej jakości.
    3. Oblicza różnicę między oryginałem a ponownie skompresowanym obrazem.
    4. Wzmacnia różnice (skalowanie jasności) i konwertuje wynik do skali szarości.

    Parametry:
        image_path (str): Ścieżka do pliku obrazu.
        quality (int): Jakość zapisu JPEG (domyślnie 90).
        scale (int): Współczynnik wzmocnienia mapy ELA (domyślnie 15).

    Zwraca:
        org_np (np.ndarray): Oryginalny obraz w formacie RGB.
        ela_gray (np.ndarray): Wzmocniona mapa ELA w skali szarości (uint8).
# Opis funkcji ela_features()
    Ekstrahuje cechy statystyczne i teksturalne z mapy ELA dla danego obrazu.

    Kroki:
    1. Wykonuje ELA i uzyskuje mapę błędu w skali szarości.
    2. Wygładza mapę ELA filtracją Gaussa, redukując szum wysokoczęstotliwościowy.
    3. Oblicza cechy statystyczne (mean, std, median, percentyle) oraz cechy relatywne.
    4. Oblicza cechy teksturalne na podstawie Laplaciana (krawędzie / zmiany intensywności).

    Parametry:
        image_path (str): Ścieżka do pliku obrazu.

    Zwraca:
        org_np (np.ndarray): Oryginalny obraz RGB.
        ela_blur (np.ndarray): Wygładzona mapa ELA (uint8).
        features (dict): Słownik cech ELA (statystyczne + teksturalne).

In [ ]:
def perform_ela_gray(image_path, quality=90, scale=15):
    try:
        with Image.open(image_path) as source_image:
            original_image = source_image.convert("RGB")
    except OSError as error:
        raise ValueError(f"Nie można odczytać obrazu dla ELA: {image_path}") from error

    # Ten sam JPEG i jakość co wcześniej, bez nadpisywania pliku na dysku.
    with BytesIO() as buffer:
        original_image.save(buffer, "JPEG", quality=quality)
        buffer.seek(0)
        with Image.open(buffer) as compressed_image:
            resaved_image = compressed_image.convert("RGB")

    ela_image = ImageChops.difference(original_image, resaved_image)
    ela_enhanced = ImageEnhance.Brightness(ela_image).enhance(scale)
    org_np = np.array(original_image)
    ela_gray = np.array(ela_enhanced.convert("L"))
    return org_np, ela_gray


def ela_features(image_path):

    org_np, ela_gray = perform_ela_gray(image_path)
    if org_np is None:
        return None, None, None

    # lekkie wygładzenie – redukcja szumu wysokoczęstotliwościowego
    ela_blur = cv.GaussianBlur(ela_gray, (5, 5), 0)

    # --- CECHY PODSTAWOWE ---
    ela_mean   = float(ela_blur.mean()) # średnia jasność mapy - ogólny poziom kompresowalności
    ela_std    = float(ela_blur.std()) # zmienność błędu
    ela_median = float(np.median(ela_blur)) # wartość środkowa, odporna na odchylenia
    ela_p90    = float(np.percentile(ela_blur, 90)) # wartość powyżej której leży 10% pikseli
    ela_p99    = float(np.percentile(ela_blur, 99)) # wskaźnik ekstremalnych anomalii

    # cecha „relatywna” – jak bardzo świeci ogon rozkładu względem tła
    ela_rel_p90 = ela_p90 - ela_median
    ela_rel_p99 = ela_p99 - ela_median

    # --- TEKSTURALNOŚĆ (Laplacian) ---
    # analiza krawędzi na mapie ELA, sfałszowane obszary mogą mieć ostre granice
    lap = cv.Laplacian(ela_blur, cv.CV_32F)
    lap_abs = np.abs(lap) # wartość bezwględna odpowiedzi filtra

    ela_lap_mean = float(lap_abs.mean())
    ela_lap_p90  = float(np.percentile(lap_abs, 90))

    features = {
        "ela_mean": ela_mean,
        "ela_std": ela_std,
        "ela_median": ela_median,
        "ela_p90": ela_p90,
        "ela_p99": ela_p99,
        "ela_rel_p90": ela_rel_p90,
        "ela_rel_p99": ela_rel_p99,
        "ela_lap_mean": ela_lap_mean,
        "ela_lap_p90": ela_lap_p90,
    }

    return org_np, ela_blur, features

# Opis funkcji perform_dct_inconsistency()
    Oblicza mapę niespójności DCT na obrazie poprzez porównanie współczynników DCT
    pomiędzy oryginałem a ponownie skompresowanym JPEG.

    Procedura:
    1. Wczytuje obraz w skali szarości i przycina go do wielokrotności block_size.
    2. Symuluje ponowną kompresję JPEG (zapisywanie + ponowne wczytanie).
    3. Dzieli oba obrazy na bloki o rozmiarze block_size x block_size.
    4. Dla każdego bloku oblicza DCT i sumuje różnice w paśmie średnich częstotliwości.
    5. Wynikowa mapa niespójności jest normalizowana do zakresu 0–255.

    Parametry:
        image_path (str): Ścieżka do obrazu.
        block_size (int): Rozmiar bloku DCT (domyślnie 8).
        quality (int): Jakość ponownej kompresji JPEG (domyślnie 90).

    Zwraca:
        img (np.ndarray): Przycięty obraz w skali szarości (uint8).
        dct_norm (np.ndarray): Znormalizowana mapa niespójności DCT (uint8, 0–255).
# Opis funkcji dct_features()
    Ekstrahuje cechy statystyczne z mapy niespójności DCT obrazu.

    Kroki:
    1. Oblicza mapę niespójności DCT (perform_dct_inconsistency).
    2. Wygładza mapę, aby zmniejszyć efekt blokowania.
    3. Wylicza cechy statystyczne (mean, std, median, percentyle) oraz cechy relatywne
       opisujące rozkład wartości na mapie.

    Parametry:
        image_path (str): Ścieżka do obrazu.

    Zwraca:
        img_gray (np.ndarray): Przycięty obraz w skali szarości.
        dct_map_smooth (np.ndarray): Wygładzona mapa niespójności DCT.
        features (dict): Słownik cech DCT (mean, std, median, p90, p99, rel_p90, rel_p99).

In [ ]:
def perform_dct_inconsistency(image_path, block_size=8, quality=90):

    img = read_dataset_image(image_path, cv.IMREAD_GRAYSCALE)

    h, w = img.shape
    h_new, w_new = (h // block_size) * block_size, (w // block_size) * block_size
    img = img[:h_new, :w_new]

    # JPEG o tej samej jakości, kodowany i odczytywany wyłącznie w pamięci.
    success, compressed = cv.imencode(
        ".jpg", img, [int(cv.IMWRITE_JPEG_QUALITY), quality]
    )
    if not success:
        raise ValueError(f"Nie można zakodować obrazu JPEG dla DCT: {image_path}")
    img2 = cv.imdecode(compressed, cv.IMREAD_GRAYSCALE)
    if img2 is None:
        raise ValueError(f"Nie można odczytać JPEG dla DCT: {image_path}")

    img2 = img2[:h_new, :w_new]

    # inicjalizacja pustej mapy niespójności o wymiarach przyciętego obrazu
    dct_map = np.zeros((h_new, w_new), dtype=np.float32)

    # iteracja po obrazie blokami o kroku block_size (8)
    for i in range(0, h_new, block_size):
        for j in range(0, w_new, block_size):
            # wyodrębnienie bloku 8x8 z oryginału (rzutowanie na float wymagane przez cv.dct)
            block1 = img[i:i+block_size, j:j+block_size].astype(np.float32)
            # wyodrębnienie odpowiadającego bloku z obrazu ponownie skompresowanego
            block2 = img2[i:i+block_size, j:j+block_size].astype(np.float32)

            # obliczenie dct dla obu bloków
            dct1 = cv.dct(block1)
            dct2 = cv.dct(block2)

            # obliczenie macierzy różnic absolutnych współczynników
            diff = np.abs(dct1 - dct2)

            # wybór pasma średnich częstotliwości; pomijamy składową stałą oraz wysokie częstotliwości
            mid_band = diff[2:6, 2:6]
            # sumowanie błędów w paśmie średnim jako miara niespójności bloku
            score = np.sum(mid_band)

            # przypisanie wyniku do całego obszaru bloku na mapie wyjściowej
            dct_map[i:i+block_size, j:j+block_size] = score

    dct_norm = cv.normalize(dct_map, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

    return img, dct_norm


def dct_features(image_path):

    img_gray, dct_norm = perform_dct_inconsistency(image_path)
    if img_gray is None:
        return None, None, None

    # lekkie wygładzenie w celu redukcji efektu wyraźnego podziału na bloki
    dct_map_smooth = cv.GaussianBlur(dct_norm, (5, 5), 0)

    # --- CECHY ---
    dct_mean   = float(dct_map_smooth.mean())
    dct_std    = float(dct_map_smooth.std())
    dct_median = float(np.median(dct_map_smooth))
    dct_p90    = float(np.percentile(dct_map_smooth, 90))
    dct_p99    = float(np.percentile(dct_map_smooth, 99))

    dct_rel_p90 = dct_p90 - dct_median
    dct_rel_p99 = dct_p99 - dct_median

    features = {
        "dct_mean": dct_mean,
        "dct_std": dct_std,
        "dct_median": dct_median,
        "dct_p90": dct_p90,
        "dct_p99": dct_p99,
        "dct_rel_p90": dct_rel_p90,
        "dct_rel_p99": dct_rel_p99,
    }

    return img_gray, dct_map_smooth, features

# Opis funkcji extract_noise_residual()
    Ekstrahuje przybliżone rezyduum szumu (PRNU-like) z obrazu.

    Proces:
    1. Konwersja obrazu do skali szarości (jeśli wejście jest RGB/BGR).
    2. Zastosowanie algorytmu odszumiania Non-local Means (fastNlMeansDenoising).
    3. Obliczenie rezyduum jako różnicy między oryginałem a obrazem odszumionym.
    4. Normalizacja rezyduum (średnia 0, odchylenie standardowe 1).

    Parametry:
        gray (np.ndarray): Obraz wejściowy w skali szarości lub RGB/BGR.

    Zwraca:
        residual (np.ndarray): Znormalizowane rezyduum szumu (float32).
# Opis funkcji noise_block_variance()
    Oblicza mapę wariancji rezyduum szumu w blokach i przeskalowuje ją do rozmiaru obrazu.

    Proces:
    1. Dzieli rezyduum na bloki o zadanym rozmiarze.
    2. Dla każdego bloku oblicza wariancję (miara lokalnej zmienności szumu).
    3. Tworzy mapę wariancji o rozdzielczości blokowej.
    4. Skalowanie mapy do pełnych wymiarów obrazu (interpolacja nearest neighbor).
    5. Normalizacja do zakresu 0–255 (uint8) w celu wizualizacji.

    Parametry:
        residual (np.ndarray): Rezyduum szumu (float32).
        block_size (int): Rozmiar bloku (domyślnie 32).

    Zwraca:
        var_norm (np.ndarray): Znormalizowana mapa wariancji (uint8, 0–255).
# Opis funkcji noise_features()
    Ekstrahuje cechy związane z szumem / PRNU z obrazu oraz generuje mapę wariancji.

    Proces:
    1. Wczytuje obraz w skali szarości.
    2. Wylicza rezyduum szumu (extract_noise_residual).
    3. Dzieli rezyduum na bloki i oblicza wariancję w każdym bloku.
    4. Oblicza globalne statystyki rozkładu wariancji (mean, std, median, percentyle).
    5. Tworzy znormalizowaną mapę wariancji do wizualizacji.

    Parametry:
        image_path (str): Ścieżka do obrazu.
        block_size (int): Rozmiar bloku dla wariancji (domyślnie 32).

    Zwraca:
        img_gray (np.ndarray): Obraz w skali szarości.
        var_norm (np.ndarray): Znormalizowana mapa wariancji (uint8).
        features (dict): Słownik cech statystycznych wariancji szumu.

In [ ]:
def extract_noise_residual(gray):
    if gray.ndim == 3:
        gray = cv.cvtColor(gray, cv.COLOR_BGR2GRAY)

    gray = gray.astype(np.uint8) # rzutowanie wymagane przez fastNlMneasDenoising
    # zastosowanie odszumiania Non-local Means
    denoised = cv.fastNlMeansDenoising(gray, h=5, # h=5 - siła odszumiania
                                       templateWindowSize=7, # rozmiar łatki do porównywania pikseli
                                       searchWindowSize=21) # obszar poszukiwania podobnych łatek wokół piksela
    gray_f = gray.astype(np.float32) # konwersja na float w celu wykonania precyzyjnego odejmowania
    denoised_f = denoised.astype(np.float32)
    residual = gray_f - denoised_f # obliczenie rezyduum szumu

    # normalizacja rezyduum; dodanie 1e-8 zapobiega dzieleniu przez zero w przypadku idealnie gładkiego obrazu
    residual -= residual.mean()
    residual /= (residual.std() + 1e-8)
    return residual


def noise_block_variance(residual, block_size=32):

    H, W = residual.shape
    height_blocks = H // block_size
    width_blocks = W // block_size

    # inicjalizacja mapy wariancji o wymiarach jeden piksel na blok
    var_map = np.zeros((height_blocks, width_blocks), dtype=np.float32)

    for by in range(height_blocks):
        for bx in range(width_blocks):
            # wycięcie odpowiedniego fragmentu rezyduum (blok_size)
            block = residual[
                by*block_size:(by+1)*block_size,
                bx*block_size:(bx+1)*block_size
            ]
            var_map[by, bx] = block.var() # oblcizenie wariancji wewnątrz bloku i przypisanie do mapy

    # przeskalowanie mapy wariancji do pełnych wymiarów obrazu (interpolacja najbliższego sąsiada)
    var_resized = cv.resize(var_map, (W, H), interpolation=cv.INTER_NEAREST)
    var_norm = cv.normalize(var_resized, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

    return var_norm


def noise_features(image_path, block_size=32):

    img_gray = read_dataset_image(image_path, cv.IMREAD_GRAYSCALE)

    residual = extract_noise_residual(img_gray)
    H, W = residual.shape

    # --- mapa wariancji w blokach (bez normalizacji) ---
    height_blocks = H // block_size
    width_blocks = W // block_size
    var_map = np.zeros((height_blocks, width_blocks), dtype=np.float32)

    for by in range(height_blocks):
        for bx in range(width_blocks):
            block = residual[
                by*block_size:(by+1)*block_size,
                bx*block_size:(bx+1)*block_size
            ]
            var_map[by, bx] = block.var()

    # spłaszczenie mapy wariancji w celu policzenia statystyk globalnych
    var_flat = var_map.ravel()

    noise_mean   = float(var_flat.mean())
    noise_std    = float(var_flat.std())
    noise_median = float(np.median(var_flat))
    noise_p90    = float(np.percentile(var_flat, 90))
    noise_p99    = float(np.percentile(var_flat, 99))

    noise_rel_p90 = noise_p90 - noise_median
    noise_rel_p99 = noise_p99 - noise_median
    noise_ratio_p90 = noise_p90 / (noise_median + 1e-6)

    features = {
        "noise_mean": noise_mean,
        "noise_std": noise_std,
        "noise_median": noise_median,
        "noise_p90": noise_p90,
        "noise_p99": noise_p99,
        "noise_rel_p90": noise_rel_p90,
        "noise_rel_p99": noise_rel_p99,
        "noise_ratio_p90": noise_ratio_p90,
    }

    # przeskalowana i znormalizowana mapa do wizualizacji
    var_resized = cv.resize(var_map, (W, H), interpolation=cv.INTER_NEAREST)
    var_norm = cv.normalize(var_resized, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

    return img_gray, var_norm, features

# Cechy
    Opis funckji extract_features()
    Ekstrahuje zbiór cech z obrazu łącząc metody ELA, DCT, analizy szumu (PRNU-like)
    oraz embeddingi CNN na patchach.

    Procedura:
    1. Oblicza cechy ELA (mapa ELA + statystyki) i dodaje je do wektora cech.
    2. Oblicza cechy DCT (mapa niespójności DCT + statystyki) i dodaje je do wektora cech.
    3. Oblicza cechy związane z rezyduum szumu (PRNU-like) i dodaje je do wektora cech.
    4. Ekstrahuje embeddingi CNN z patchy (np. 3×3), agreguje je w wektor 1024-wymiarowy
       i dołącza do końca wektora cech.

    Zwracane wartości:
        feature_vector (np.ndarray): Jednowymiarowy wektor cech (float32) złączonych
                                     z wszystkich metod.
        feature_names (list[str]): Lista nazw cech w tej samej kolejności co feature_vector.
        all_features (dict): Słownik zawierający podział cech na grupy:
                             'ela', 'dct', 'noise', 'cnn' (z metadanymi).

In [ ]:
def extract_features(image_path, grid=(3, 3)):
    feature_values = []
    feature_names = []
    all_features = {}

    # --- ELA ---
    org_np, ela_map, ela_feat = ela_features(image_path)
    if ela_feat is None:
        return None, None, None
    # zachowaj kolejność deterministyczną
    for k in sorted(ela_feat.keys()):
        feature_values.append(float(ela_feat[k]))
        feature_names.append(f"ela_{k}")
    all_features["ela"] = ela_feat

    # --- DCT ---
    img_gray, dct_map, dct_feat = dct_features(image_path)
    if dct_feat is None:
        return None, None, None
    for k in sorted(dct_feat.keys()):
        feature_values.append(float(dct_feat[k]))
        feature_names.append(f"dct_{k}")
    all_features["dct"] = dct_feat

    # --- NOISE / PRNU-like ---
    img_gray2, var_norm, noise_feat = noise_features(image_path)
    if noise_feat is None:
        return None, None, None
    for k in sorted(noise_feat.keys()):
        feature_values.append(float(noise_feat[k]))
        feature_names.append(f"noise_{k}")
    all_features["noise"] = noise_feat

    # --- CNN PATCH 3x3 (1024) ---
    cnn_vals, cnn_names, cnn_meta = extract_features_patch_cnn(image_path, grid=grid)
    # cnn_vals to lista z jednym np.ndarray(1024,)
    cnn_feat = cnn_vals[0]
    feature_values.extend(cnn_feat.tolist())
    feature_names.extend(cnn_names)
    all_features.update(cnn_meta)

    feature_vector = np.array(feature_values, dtype=np.float32)
    return feature_vector, feature_names, all_features

# cechy po wszystkich zdjeciach
    Funkcja zbiera ścieżki do obrazów z dwóch katalogów (real i tampered),
    przypisuje im etykiety binarne (0 = real, 1 = tampered) oraz zapisuje nazwy plików.
    Następnie wywołuje funkcję, tworząc listy ścieżek, tablicę etykiet i listę nazw plików,
    a na końcu wypisuje podstawowe informacje o zbiorze (liczba obrazów, etykiety, przykładowe pliki).

In [ ]:
import pandas as pd


def get_image_paths_and_labels(real_folder, tampered_folder):
    image_paths = []
    labels = []  # 0 = real, 1 = tampered; etykiety nie są cechami.
    names = []
    for label, folder in enumerate((real_folder, tampered_folder)):
        for path in list_dataset_images(folder):
            image_paths.append(str(path))
            labels.append(label)
            names.append(path.name)
    return image_paths, np.array(labels, dtype=int), names


image_paths, y, filenames = get_image_paths_and_labels(real_folder, tampered_folder)
print("Liczba obrazów:", len(image_paths))
print("Etykiety (0=real,1=tampered):", y)
print("Przykładowe pliki:", filenames[:5])

#   Ekstrakcja wektorów cech dla wszystkich obrazów w zbiorze.
Dla każdego obrazu wywoływana jest funkcja extract_features, która zwraca:
  - fv: wektor cech (np.array)
  - fnames: lista nazw cech (ta sama dla wszystkich obrazów)
  - feats_dict: słownik cech pogrupowanych wg metody (ELA, DCT, noise, CNN)<br>
Następnie wektory cech są łączone w macierz X (n_samples x n_features),<br>
a nazwy cech są zapisane tylko raz, przy pierwszym obrazie.<br>
Na końcu wypisywany jest kształt macierzy cech oraz lista nazw cech.<br>

In [ ]:
feature_vectors = []
feature_names = None
all_feature_dicts = []   # opcjonalnie, do debugowania

for idx, path in enumerate(image_paths):
    fv, fnames, feats_dict = extract_features(path)
    if fv is None:
        raise RuntimeError(f"Nie udało się wyciągnąć cech dla {path}")

    feature_vectors.append(fv)
    all_feature_dicts.append(feats_dict)

    if feature_names is None:
        feature_names = fnames  # ta sama kolejność dla wszystkich

X = np.vstack(feature_vectors)  # shape: (n_samples, n_features)

print("Kształt X:", X.shape)
print("Nazwy cech:")
for name in feature_names:
    print("  ", name)


In [ ]:
df_features = pd.DataFrame(X, columns=feature_names)
df_features["label"] = y
df_features["filename"] = filenames

df_features.head(30)


# LOOCV
Przeprowadzenie walidacji Leave-One-Out (LOOCV) dla klasyfikatora Gradient Boosting.
Dla każdej iteracji:
  1) jeden obraz jest używany jako próbka testowa, a pozostałe jako treningowe,
  2) model pipeline (standaryzacja + GradientBoosting) jest trenowany na zbiorze treningowym,
  3) dla próbki testowej obliczane jest prawdopodobieństwo przynależności do klasy
     "tampered" (1) oraz decyzja klasyfikacyjna (threshold 0.5),
  4) wyniki (prawdziwa etykieta, predykcja, prawdopodobieństwo, nazwa pliku)
     są zapisywane do list.
Po zakończeniu pętli wypisywana jest informacja o liczbie przeprowadzonych iteracji.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

loo = LeaveOneOut()

y_true = []
y_pred = []
y_prob = []     # prawdopodobieństwo klasy "tampered" (1)
test_files = []

# model: skalowanie + gradient boosting
base_model = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(random_state=42, learning_rate=0.1),
    memory=None,  # bez cache na dysku; jawnie zachowuje poprzednią wartość domyślną
)

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = clone(base_model)  # niezależny pipeline w każdej iteracji
    model.fit(X_train, y_train)

    # predykcja
    prob = model.predict_proba(X_test)[0, 1]   # P(klasa=1)
    pred = int(prob >= 0.5)

    y_true.append(int(y_test[0]))
    y_pred.append(pred)
    y_prob.append(prob)
    test_files.append(filenames[test_idx[0]])

print(f"Gotowe LOOCV dla {len(y_true)} obrazów.")

Utworzenie tabeli wyników z przeprowadzonego LOOCV. <br>
Dane z list (nazwa pliku, etykieta prawdziwa, etykieta predykowana, prawdopodobieństwo) <br>
są zapisywane do DataFrame. Następnie dodawane są czytelne nazwy klas <br>
('real' / 'tampered') dla etykiet prawdziwych i predykowanych. <br>
Na końcu tabela jest sortowana po nazwie pliku i indeksowana od zera. <br>

In [ ]:
results_df = pd.DataFrame({
    "filename": test_files,
    "true_label": y_true,
    "pred_label": y_pred,
    "prob_tampered": y_prob
})

# dla czytelności 'predicted_class_name'
def label_to_name(lbl):
    return "tampered" if lbl == 1 else "real"

results_df["true_name"] = results_df["true_label"].apply(label_to_name)
results_df["pred_name"] = results_df["pred_label"].apply(label_to_name)

results_df = results_df.sort_values("filename").reset_index(drop=True)

results_df


# Obliczenie podstawowych metryk klasyfikacji na podstawie wyników LOOCV:
    - Accuracy: odsetek poprawnie sklasyfikowanych obrazów
    - F1-score: zbalansowana miara uwzględniająca precyzję i recall
    - Confusion matrix: macierz pomyłek (wiersze = prawdziwe klasy, kolumny = predykcje)
    - Classification report: szczegółowy raport (precision, recall, f1, support) dla klas 'real' i 'tampered'
    Wypisanie wyników.

In [ ]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("Accuracy:", acc)
print("F1-score:", f1)
print("Confusion matrix (rows=true, cols=pred):")
print(cm)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["real", "tampered"]))


Wypisanie danych obrazów, których nie zdołano rozpoznać.

In [ ]:
wrong_df = results_df.copy()
wrong_df = wrong_df[wrong_df['true_label'] != wrong_df['pred_label']]
wrong_df

In [ ]:
def build_full_path(row):
    folder = tampered_folder if row["true_label"] == 1 else real_folder
    return str(Path(folder) / row["filename"])


# Lista pusta także poprawnie tworzy kolumnę, gdy nie ma błędnych predykcji.
wrong_df["full_path"] = [
    build_full_path(row) for _, row in wrong_df.iterrows()
]
wrong_df

Wyświetlenie nierozpoznanych obrazów wraz z ich danymi.

In [ ]:
def show_misclassifications(results):
    if results.empty:
        print("Brak błędnych klasyfikacji.")
        return None
    images = []
    titles = []
    for _, row in results.iterrows():
        image = read_dataset_image(row["full_path"])
        images.append(cv.cvtColor(image, cv.COLOR_BGR2RGB))
        titles.append(
            f"{row['filename']}\n"
            f"True: {row['true_label']} | Pred: {row['pred_label']} | "
            f"P(tampered): {row['prob_tampered']:.3f}"
        )
    return show_image_grid(images, titles)


show_misclassifications(wrong_df)

## Historyczne wnioski z pierwotnego eksperymentu

Poniższe obserwacje zachowano jako zapis wcześniejszego przebiegu na 30 lokalnych obrazach (10 real, 20 tampered). Nie są aktualnym raportem walidacji, dowodem skuteczności produkcyjnej ani potwierdzeniem uogólniania na inne źródła zdjęć. Podane wyjaśnienia błędów są hipotezami, nie ustalonymi przyczynami. Mały, niezbalansowany zbiór i możliwe powiązania między obrazami ograniczają wiarygodność LOOCV. Nazwy plików nie są cechami modelu.

### 1. Ocena ogólna (Metryki)

W pierwotnym przebiegu model osiągnął accuracy na poziomie 93%, co przy tak małym zbiorze testowym (30 zdjęć) jest wynikiem obiecującym, jednak wymagającym ostrożności.
Jednym ze znaczących problemów jest brak zbalansowania zbiorów. W dyspozycji jest 20 zdjęć zmanipulowanych i tylko 10 prawdziwych. Wobec takiej dysproporcji model ma tendencję do "zgadywania" klasy większościowej.
Potwierdzają to wyniki recall: 0.95 - fake i 0.90 - real.
Zastanawiająca jest pewność siebie modelu. Należy zwrócić uwagę, że dla przypadków, gdzie model się myli, nadal jest bardzo pewny siebie (osiąganie prawdopodobieństw bliskich 0% lub 100%).
Takie wyniki wskazują na to, że model nie "wahał się", lecz natrafił na wzorce, które zupełnie go zmyliły.
Model działa bez zarzutów dla oczywistych manipulacji, jednak błędnie interpretuje naturalny szum jako fałszerstwo i nie "docenia" gładkich manipulacji geometrycznych.
### 2. Analiza błędów
Przeanalizowano dwa kluczowe błędy widoczne na obrazach niesklasyfikowanych w kontekście wygenerowanych cech. <br>
Dla obrazu przedstawiającego chłopca na trawie model był pewny na 99.99%, że jest to fałszerstwo.
Najprawdopodobniejszą przyczyną jest szum, tworzony przez trawę. W analizie Error Level Analysis trawa, liście, i żwir generują bardzo dużo wysokoczęstotliwościowego szumu. Szum ten może być łatwo pomylony z artefaktami kompresji powstałymi przy wklejaniu obiektów. Dowodem tej hipotezy jest bardzo duża wartość średniej transformaty kosinusowej dla tego obrazu, podczas gdy dla innych jest ok. dwukrotnie mniejsza. Wysoka wartość DCT jest typowym objawem manipulacji zdjęcia. Model uznał to za ślad manipulacji. <br>
Obraz zmanipulowany, przedstawiający fasadę bloku, został błędnie zaklasyfikowany jako prawdziwy. Model w tym przypadku był pewny na 99.99%, że jest to oryginał.
Manipulacja z rodzaju "transform" na obiektach geometrycznych jest bardzo trudna do wykrycia. Linie proste i powtarzalne wzorce maskują niedoskonałości edycji.
Dowodem tej hipotezy są wartości cech ela_ela_mean dla tego zdjęcia. Są one bardzo przeciętne, co sprawia, że obraz pod tym względem nie wyróżnia się na tle pozostałych zdjęć.
### 3. Wnioski z danych statystycznych
Analiza tabeli cech pokazuje, że proste statystyki (jak średnia ELA) nie są wystarczające do separacji klas. Zdjęcia prawdziwe cechują się ela_ela_mean w zakresie ok. 2.4 - 6.8.
Zdjęcia zmanipulowane mają ten zakres szerszy, ale mocno pokrywający się: 1.6 - 8.5.
Wskazuje to na to, że wytrenowany model słusznie polega na bardziej złożonych cechach ("cnn_patch"), ale w przypadku obrazu "chłopca z psami" nawet te głębokie cechy są myląco podobne do tych z fałszywek.
### 4. Propozycje udoskonalenia modelu.
Dodanie do zbioru treningowego większej liczby zdjęć prawdziwych, szczególnie tych o wysokiej entropii (np. trawa, piasek, deszcz). Model musi zrozumieć, że duża liczba detali na obrazie nie jest wyznacznikiem fałszerstwa.
W celu poprawienia wykrywania manipulacji na budynkach, należy trenować model na zdjęciach, które zostały tylko delikatnie przeskalowane lub obrócone i oznaczyć je jako zmanipulowane (jeśli celem jest wykrywanie każdej ingerencji) lub uczyć go na parach (oryginał w kontrze do delikatnej transformacji). Dodatkowo w przypadku zdjęcia z blokiem, metody oparte na analizie częstotliwościowej DCT zawiodły. Warto rozważyć dodanie do modelu modułu sprawdzającego spójność perspektywy albo cieni, aby skutecznie wykrywać manipulacje geometryczne.